# Packet P-010 — Panel Replacement for Device 1064
**Trigger:** P-009 found Device 1064 fragile under ±10% feature noise (30% top-20 rate vs 100%/96% for other panel members).
**Goal:** Find a perturbation-robust MA-FA mixed candidate to replace Device 1064.
**Method:** Screen all MA-FA devices from the P-005 qualified pool, test perturbation robustness, select the best.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('/Users/johnodowd/Projects/materia-arche/perovskite_stability_clean.csv')

FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'

X = df[FEATURES].copy()
y = np.log1p(df[TARGET])

# Identify MA-FA mixed devices: MA > 0 AND FA > 0 AND Cs == 0
ma_fa_mask = (df['MA'] > 0) & (df['FA'] > 0) & (df['Cs'] == 0)
print(f"Total devices in dataset: {len(df)}")
print(f"MA-FA mixed devices (MA>0, FA>0, Cs==0): {ma_fa_mask.sum()}")

# Filter to T80 >= 500 hours (P-005 threshold)
t80_mask = df[TARGET] >= 500
ma_fa_qualified = df.index[ma_fa_mask & t80_mask].tolist()
print(f"MA-FA mixed with T80 >= 500h: {len(ma_fa_qualified)}")
print(f"\nMA-FA candidate indices: {ma_fa_qualified}")
for idx in ma_fa_qualified:
    row = df.loc[idx]
    print(f"  Device {idx}: MA={row['MA']:.3f}, FA={row['FA']:.3f}, Cs={row['Cs']:.3f}, "
          f"T80={row[TARGET]:.1f}h")

Total devices in dataset: 1543
MA-FA mixed devices (MA>0, FA>0, Cs==0): 197
MA-FA mixed with T80 >= 500h: 67

MA-FA candidate indices: [35, 36, 72, 74, 115, 131, 196, 200, 204, 260, 265, 266, 267, 300, 348, 350, 390, 403, 405, 415, 445, 449, 458, 474, 480, 506, 548, 557, 585, 591, 609, 610, 621, 666, 673, 760, 885, 938, 950, 954, 963, 994, 1044, 1052, 1053, 1054, 1057, 1063, 1064, 1065, 1066, 1208, 1298, 1302, 1309, 1310, 1311, 1318, 1319, 1320, 1327, 1345, 1346, 1416, 1417, 1430, 1474]
  Device 35: MA=0.952, FA=0.048, Cs=0.000, T80=720.0h
  Device 36: MA=0.150, FA=0.850, Cs=0.000, T80=1200.0h
  Device 72: MA=0.500, FA=0.500, Cs=0.000, T80=800.0h
  Device 74: MA=0.250, FA=0.750, Cs=0.000, T80=600.0h
  Device 115: MA=0.170, FA=0.830, Cs=0.000, T80=1000.0h
  Device 131: MA=0.100, FA=0.900, Cs=0.000, T80=1500.0h
  Device 196: MA=0.170, FA=0.830, Cs=0.000, T80=2300.0h
  Device 200: MA=0.150, FA=0.850, Cs=0.000, T80=672.0h
  Device 204: MA=0.150, FA=0.850, Cs=0.000, T80=1560.0h
  Device 260

In [2]:
# Cell 3: Cross-split robustness screening (P-005 method)
# 20 random splits, track top-20 rate for each MA-FA candidate

from collections import defaultdict

n_splits = 20
top_k = 20

# Track stats per device
device_stats = defaultdict(lambda: {'appearances': 0, 'top20_count': 0, 'ranks': []})

for split_seed in range(n_splits):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=split_seed
    )
    
    model = ExtraTreesRegressor(
        n_estimators=700, max_features='sqrt', min_samples_split=3,
        min_samples_leaf=1, bootstrap=False, random_state=42
    )
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    test_indices = X_test.index.tolist()
    
    # Rank by predicted value (descending = rank 1 is highest predicted stability)
    ranked = sorted(zip(test_indices, preds), key=lambda x: -x[1])
    rank_map = {idx: rank+1 for rank, (idx, _) in enumerate(ranked)}
    top20_set = set([idx for idx, _ in ranked[:top_k]])
    
    # Track MA-FA candidates that appeared in this test set
    for dev in ma_fa_qualified:
        if dev in rank_map:
            device_stats[dev]['appearances'] += 1
            device_stats[dev]['ranks'].append(rank_map[dev])
            if dev in top20_set:
                device_stats[dev]['top20_count'] += 1

# Filter: >=80% top-20 rate AND >=3 appearances
print("=== Cross-Split Screening Results (MA-FA candidates) ===\n")
print(f"{'Device':>8} {'Appear':>7} {'Top20':>6} {'Rate':>7} {'MeanRank':>9} {'Status':>10}")
print("-" * 55)

qualified_candidates = []
for dev in ma_fa_qualified:
    s = device_stats[dev]
    if s['appearances'] == 0:
        continue
    rate = s['top20_count'] / s['appearances']
    mean_rank = np.mean(s['ranks'])
    status = "QUALIFIED" if (rate >= 0.80 and s['appearances'] >= 3) else "filtered"
    print(f"{dev:>8} {s['appearances']:>7} {s['top20_count']:>6} {rate:>7.1%} {mean_rank:>9.1f} {status:>10}")
    if rate >= 0.80 and s['appearances'] >= 3:
        qualified_candidates.append(dev)

print(f"\nQualified MA-FA candidates (>=80% top-20, >=3 appearances): {len(qualified_candidates)}")
print(f"Indices: {qualified_candidates}")

# Fallback: if no MA-FA candidates qualify, relax criteria
if len(qualified_candidates) == 0:
    print("\n*** No MA-FA candidates met >=80% threshold. Relaxing to >=50% ***")
    for dev in ma_fa_qualified:
        s = device_stats[dev]
        if s['appearances'] >= 3:
            rate = s['top20_count'] / s['appearances']
            if rate >= 0.50:
                qualified_candidates.append(dev)
    print(f"Relaxed qualified: {qualified_candidates}")

if len(qualified_candidates) == 0:
    print("\n*** Still none. Will expand to ALL non-MA-only, non-FA-Cs devices ***")
    # Expand search
    expanded_mask = ~((df['MA'] > 0) & (df['FA'] == 0) & (df['Cs'] == 0)) & \
                    ~((df['FA'] > 0) & (df['Cs'] > 0) & (df['MA'] == 0)) & \
                    t80_mask
    expanded_candidates = df.index[expanded_mask].tolist()
    # Remove devices 850, 119
    expanded_candidates = [d for d in expanded_candidates if d not in [850, 119]]
    print(f"Expanded candidate pool: {len(expanded_candidates)} devices")
    # Re-run screening for expanded set
    device_stats_exp = defaultdict(lambda: {'appearances': 0, 'top20_count': 0, 'ranks': []})
    for split_seed in range(n_splits):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=split_seed
        )
        model = ExtraTreesRegressor(
            n_estimators=700, max_features='sqrt', min_samples_split=3,
            min_samples_leaf=1, bootstrap=False, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        test_indices = X_test.index.tolist()
        ranked = sorted(zip(test_indices, preds), key=lambda x: -x[1])
        rank_map = {idx: rank+1 for rank, (idx, _) in enumerate(ranked)}
        top20_set = set([idx for idx, _ in ranked[:top_k]])
        for dev in expanded_candidates:
            if dev in rank_map:
                device_stats_exp[dev]['appearances'] += 1
                device_stats_exp[dev]['ranks'].append(rank_map[dev])
                if dev in top20_set:
                    device_stats_exp[dev]['top20_count'] += 1
    for dev in expanded_candidates:
        s = device_stats_exp[dev]
        if s['appearances'] >= 3:
            rate = s['top20_count'] / s['appearances']
            if rate >= 0.80:
                qualified_candidates.append(dev)
    print(f"Expanded qualified candidates: {len(qualified_candidates)}")

=== Cross-Split Screening Results (MA-FA candidates) ===

  Device  Appear  Top20    Rate  MeanRank     Status
-------------------------------------------------------
      35       3      0    0.0%      68.7   filtered
      36       1      0    0.0%      31.0   filtered
      72       3      0    0.0%      49.7   filtered
      74       3      0    0.0%     252.3   filtered
     115       4      0    0.0%      69.2   filtered
     131       4      0    0.0%     121.5   filtered
     196       3      0    0.0%      37.3   filtered
     200       2      0    0.0%      69.0   filtered
     204       8      0    0.0%      89.8   filtered
     260       2      0    0.0%      71.5   filtered
     265       4      4  100.0%      12.5  QUALIFIED
     266       3      3  100.0%      10.0  QUALIFIED
     267       7      7  100.0%       6.0  QUALIFIED
     300       2      0    0.0%      89.0   filtered
     348       3      0    0.0%      74.0   filtered
     350       4      0    0.0%      8

In [3]:
# Cell 4: Perturbation robustness test (P-009 method)
# For each qualified candidate, test perturbation robustness across splits

noise_level = 0.10
n_trials = 50

perturbation_results = []

for dev in qualified_candidates:
    top20_hits_clean = 0
    top20_hits_noisy = 0
    total_appearances = 0
    clean_ranks = []
    noisy_ranks = []
    
    for split_seed in range(n_splits):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=split_seed
        )
        
        if dev not in X_test.index:
            continue
        
        total_appearances += 1
        
        model = ExtraTreesRegressor(
            n_estimators=700, max_features='sqrt', min_samples_split=3,
            min_samples_leaf=1, bootstrap=False, random_state=42
        )
        model.fit(X_train, y_train)
        
        # Clean ranking
        preds_clean = model.predict(X_test)
        test_indices = X_test.index.tolist()
        ranked_clean = sorted(zip(test_indices, preds_clean), key=lambda x: -x[1])
        rank_map_clean = {idx: rank+1 for rank, (idx, _) in enumerate(ranked_clean)}
        clean_ranks.append(rank_map_clean[dev])
        if rank_map_clean[dev] <= top_k:
            top20_hits_clean += 1
        
        # Perturbation trials
        trial_top20 = 0
        trial_ranks = []
        for trial in range(n_trials):
            np.random.seed(split_seed * 1000 + trial)
            X_test_noisy = X_test.copy()
            noise = np.random.uniform(1 - noise_level, 1 + noise_level, X_test_noisy.shape)
            X_test_noisy = X_test_noisy * noise
            
            preds_noisy = model.predict(X_test_noisy)
            ranked_noisy = sorted(zip(test_indices, preds_noisy), key=lambda x: -x[1])
            rank_map_noisy = {idx: rank+1 for rank, (idx, _) in enumerate(ranked_noisy)}
            trial_ranks.append(rank_map_noisy[dev])
            if rank_map_noisy[dev] <= top_k:
                trial_top20 += 1
        
        noisy_ranks.extend(trial_ranks)
        top20_hits_noisy += trial_top20
    
    if total_appearances > 0:
        clean_top20_rate = top20_hits_clean / total_appearances
        total_noisy_trials = total_appearances * n_trials
        noisy_top20_rate = top20_hits_noisy / total_noisy_trials
        mean_clean_rank = np.mean(clean_ranks)
        mean_noisy_rank = np.mean(noisy_ranks)
        
        perturbation_results.append({
            'device': dev,
            'appearances': total_appearances,
            'clean_top20_rate': clean_top20_rate,
            'mean_clean_rank': mean_clean_rank,
            'noisy_top20_rate': noisy_top20_rate,
            'mean_noisy_rank': mean_noisy_rank,
        })

# Sort by noisy top-20 rate (desc), then mean noisy rank (asc)
perturbation_results.sort(key=lambda x: (-x['noisy_top20_rate'], x['mean_noisy_rank']))

print("=== Perturbation Robustness Leaderboard (10% noise, 50 trials/split) ===\n")
print(f"{'Device':>8} {'Appear':>7} {'CleanT20':>9} {'CleanRank':>10} {'NoisyT20':>9} {'NoisyRank':>10}")
print("-" * 60)
for r in perturbation_results:
    print(f"{r['device']:>8} {r['appearances']:>7} {r['clean_top20_rate']:>9.1%} "
          f"{r['mean_clean_rank']:>10.1f} {r['noisy_top20_rate']:>9.1%} {r['mean_noisy_rank']:>10.1f}")

if perturbation_results:
    best = perturbation_results[0]
    print(f"\n>>> Best MA-FA replacement candidate: Device {best['device']}")
    print(f"    Noisy top-20 rate: {best['noisy_top20_rate']:.1%}")
    print(f"    Mean noisy rank: {best['mean_noisy_rank']:.1f}")
else:
    print("\n*** No candidates survived perturbation testing ***")

=== Perturbation Robustness Leaderboard (10% noise, 50 trials/split) ===

  Device  Appear  CleanT20  CleanRank  NoisyT20  NoisyRank
------------------------------------------------------------
    1320       3    100.0%        6.3    100.0%        5.5
     267       7    100.0%        6.0     98.9%        5.3
    1319       3    100.0%        7.0     98.7%        9.1
     266       3    100.0%       10.0     96.0%        9.3
     610       5    100.0%       13.0     94.4%        8.2
     265       4    100.0%       12.5     94.0%       10.9
    1066       6     83.3%        6.8     67.0%       22.8
    1063       5     80.0%       10.8     64.8%       24.6
    1064       5     80.0%        7.8     58.8%       24.4

>>> Best MA-FA replacement candidate: Device 1320
    Noisy top-20 rate: 100.0%
    Mean noisy rank: 5.5


In [4]:
# Cell 5: Select replacement and verify panel

# Also run perturbation test for panel anchors (850, 119) for comparison
panel_anchors = [850, 119]
anchor_results = {}

for dev in panel_anchors:
    top20_hits_clean = 0
    top20_hits_noisy = 0
    total_appearances = 0
    clean_ranks = []
    noisy_ranks = []
    
    for split_seed in range(n_splits):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=split_seed
        )
        if dev not in X_test.index:
            continue
        total_appearances += 1
        
        model = ExtraTreesRegressor(
            n_estimators=700, max_features='sqrt', min_samples_split=3,
            min_samples_leaf=1, bootstrap=False, random_state=42
        )
        model.fit(X_train, y_train)
        
        preds_clean = model.predict(X_test)
        test_indices = X_test.index.tolist()
        ranked_clean = sorted(zip(test_indices, preds_clean), key=lambda x: -x[1])
        rank_map_clean = {idx: rank+1 for rank, (idx, _) in enumerate(ranked_clean)}
        clean_ranks.append(rank_map_clean[dev])
        if rank_map_clean[dev] <= top_k:
            top20_hits_clean += 1
        
        trial_top20 = 0
        trial_ranks_list = []
        for trial in range(n_trials):
            np.random.seed(split_seed * 1000 + trial)
            X_test_noisy = X_test.copy()
            noise = np.random.uniform(1 - noise_level, 1 + noise_level, X_test_noisy.shape)
            X_test_noisy = X_test_noisy * noise
            preds_noisy = model.predict(X_test_noisy)
            ranked_noisy = sorted(zip(test_indices, preds_noisy), key=lambda x: -x[1])
            rank_map_noisy = {idx: rank+1 for rank, (idx, _) in enumerate(ranked_noisy)}
            trial_ranks_list.append(rank_map_noisy[dev])
            if rank_map_noisy[dev] <= top_k:
                trial_top20 += 1
        
        noisy_ranks.extend(trial_ranks_list)
        top20_hits_noisy += trial_top20
    
    if total_appearances > 0:
        anchor_results[dev] = {
            'appearances': total_appearances,
            'clean_top20_rate': top20_hits_clean / total_appearances,
            'mean_clean_rank': np.mean(clean_ranks),
            'noisy_top20_rate': top20_hits_noisy / (total_appearances * n_trials),
            'mean_noisy_rank': np.mean(noisy_ranks),
        }

# Select best replacement
if perturbation_results:
    replacement = perturbation_results[0]
    replacement_dev = replacement['device']
else:
    replacement_dev = None

print("=" * 70)
print("          NEW PANEL (replacing Device 1064)")
print("=" * 70)

panel_devices = [
    (850, 'MA-only'),
    (replacement_dev, 'MA-FA mixed (NEW)') if replacement_dev else (None, 'MA-FA mixed (NONE)'),
    (119, 'FA-Cs'),
]

comp_cols = ['MA', 'FA', 'Cs', 'Pb', 'I', 'Br']

for dev, family in panel_devices:
    if dev is None:
        print(f"\n  {family}: NO REPLACEMENT FOUND")
        continue
    
    row = df.loc[dev]
    
    # Get stats
    if dev in anchor_results:
        stats = anchor_results[dev]
    elif replacement_dev and dev == replacement_dev:
        stats = replacement
    else:
        stats = {'clean_top20_rate': 'N/A', 'mean_clean_rank': 'N/A',
                 'noisy_top20_rate': 'N/A', 'mean_noisy_rank': 'N/A', 'appearances': 'N/A'}
    
    print(f"\n  Device {dev} ({family}):")
    print(f"    T80 = {row[TARGET]:.1f} hours")
    print(f"    Composition: MA={row['MA']:.3f}, FA={row['FA']:.3f}, Cs={row['Cs']:.3f}, "
          f"Pb={row['Pb']:.3f}, I={row['I']:.3f}, Br={row['Br']:.3f}")
    if isinstance(stats.get('clean_top20_rate'), float):
        print(f"    Clean top-20 rate: {stats['clean_top20_rate']:.1%}, mean rank: {stats['mean_clean_rank']:.1f}")
        print(f"    Noisy top-20 rate (10%): {stats['noisy_top20_rate']:.1%}, mean noisy rank: {stats['mean_noisy_rank']:.1f}")
        print(f"    Test-set appearances: {stats['appearances']}/{n_splits}")

print("\n" + "=" * 70)
if replacement_dev:
    print(f"  REPLACEMENT: Device {replacement_dev} replaces Device 1064")
    print(f"  Improvement: 1064 had 30% noisy top-20 rate → {replacement['noisy_top20_rate']:.1%}")
else:
    print("  NO SUITABLE REPLACEMENT FOUND")

          NEW PANEL (replacing Device 1064)

  Device 850 (MA-only):
    T80 = 3400.0 hours
    Composition: MA=3.000, FA=0.000, Cs=0.000, Pb=4.000, I=13.000, Br=0.000
    Clean top-20 rate: 100.0%, mean rank: 1.0
    Noisy top-20 rate (10%): 100.0%, mean noisy rank: 1.4
    Test-set appearances: 4/20

  Device 1320 (MA-FA mixed (NEW)):
    T80 = 940.0 hours
    Composition: MA=0.250, FA=0.750, Cs=0.000, Pb=1.000, I=2.770, Br=0.250
    Clean top-20 rate: 100.0%, mean rank: 6.3
    Noisy top-20 rate (10%): 100.0%, mean noisy rank: 5.5
    Test-set appearances: 3/20

  Device 119 (FA-Cs):
    T80 = 3423.0 hours
    Composition: MA=0.000, FA=0.830, Cs=0.170, Pb=1.000, I=1.800, Br=1.200
    Clean top-20 rate: 100.0%, mean rank: 9.5
    Noisy top-20 rate (10%): 100.0%, mean noisy rank: 7.3
    Test-set appearances: 4/20

  REPLACEMENT: Device 1320 replaces Device 1064
  Improvement: 1064 had 30% noisy top-20 rate → 100.0%


In [5]:
# Cell 6: Save results CSV

# Build screening results table
screening_rows = []
for r in perturbation_results:
    dev = r['device']
    row = df.loc[dev]
    screening_rows.append({
        'device_index': dev,
        'family': 'MA-FA mixed',
        'MA': row['MA'], 'FA': row['FA'], 'Cs': row['Cs'],
        'Pb': row['Pb'], 'I': row['I'], 'Br': row['Br'],
        'T80_hours': row[TARGET],
        'test_appearances': r['appearances'],
        'clean_top20_rate': round(r['clean_top20_rate'], 3),
        'mean_clean_rank': round(r['mean_clean_rank'], 1),
        'noisy_top20_rate_10pct': round(r['noisy_top20_rate'], 3),
        'mean_noisy_rank': round(r['mean_noisy_rank'], 1),
        'selected_for_panel': dev == replacement_dev,
    })

# Add panel anchors
for dev, family in [(850, 'MA-only'), (119, 'FA-Cs')]:
    row = df.loc[dev]
    stats = anchor_results.get(dev, {})
    screening_rows.append({
        'device_index': dev,
        'family': family,
        'MA': row['MA'], 'FA': row['FA'], 'Cs': row['Cs'],
        'Pb': row['Pb'], 'I': row['I'], 'Br': row['Br'],
        'T80_hours': row[TARGET],
        'test_appearances': stats.get('appearances', 'N/A'),
        'clean_top20_rate': round(stats['clean_top20_rate'], 3) if 'clean_top20_rate' in stats else 'N/A',
        'mean_clean_rank': round(stats['mean_clean_rank'], 1) if 'mean_clean_rank' in stats else 'N/A',
        'noisy_top20_rate_10pct': round(stats['noisy_top20_rate'], 3) if 'noisy_top20_rate' in stats else 'N/A',
        'mean_noisy_rank': round(stats['mean_noisy_rank'], 1) if 'mean_noisy_rank' in stats else 'N/A',
        'selected_for_panel': True,  # anchors always selected
    })

results_df = pd.DataFrame(screening_rows)
out_path = '/Users/johnodowd/Projects/materia-arche/Packet_P010_Panel_Replacement.csv'
results_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")
print(f"Rows: {len(results_df)}")
print(results_df.to_string(index=False))

Saved to /Users/johnodowd/Projects/materia-arche/Packet_P010_Panel_Replacement.csv
Rows: 11
 device_index      family   MA   FA   Cs  Pb     I   Br  T80_hours  test_appearances  clean_top20_rate  mean_clean_rank  noisy_top20_rate_10pct  mean_noisy_rank  selected_for_panel
         1320 MA-FA mixed 0.25 0.75 0.00 1.0  2.77 0.25      940.0                 3             1.000              6.3                   1.000              5.5                True
          267 MA-FA mixed 0.40 0.60 0.00 1.0  3.00 0.00      600.0                 7             1.000              6.0                   0.989              5.3               False
         1319 MA-FA mixed 0.25 0.75 0.00 1.0  2.76 0.25      860.0                 3             1.000              7.0                   0.987              9.1               False
          266 MA-FA mixed 0.40 0.60 0.00 1.0  3.00 0.00     1350.0                 3             1.000             10.0                   0.960              9.3               False
   

## Status

**Decision logic:**
- If replacement found with >=80% top-20 rate at 10% noise: **Confirmed / Advance** — FINAL panel lock
- If best candidate is 60-80%: **Promising / Iterate** — may need further screening
- If no MA-FA candidate passes 60%: Consider dropping MA-FA family requirement

**Next:** If Confirmed, this panel proceeds to P-011+ for deeper analysis. The panel is now locked.